<a href="https://colab.research.google.com/github/DrStephPowers/BIA/blob/main/Ch7_Dashboard_Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import pandas as pd

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Youtube Data (from Web Analytics)

In [25]:
TotalViews = pd.read_csv('/content/drive/MyDrive/BUS 4640 - Stephanie/Workbooks/DATA/Totalviews.csv')

In [26]:
TotalViews.head()

,Date,Views
0,2013-09-22,44
1,2013-09-23,2
2,2013-09-24,27
3,2013-09-25,48
4,2013-09-26,103


In [27]:
age = pd.read_csv('/content/drive/MyDrive/BUS 4640 - Stephanie/Workbooks/DATA/age.csv')

In [28]:
age.head()

,Viewer age,Views (%),Average view duration,Average percentage viewed (%),Watch time (hours) (%)
0,13–17 years,0.04,0:03:08,40.92,0.05
1,18–24 years,49.01,0:02:26,25.27,43.04
2,25–34 years,34.09,0:02:55,30.96,35.84
3,35–44 years,13.75,0:03:33,36.51,17.55
4,45–54 years,2.99,0:03:09,33.15,3.38


In [29]:
gender = pd.read_csv('/content/drive/MyDrive/BUS 4640 - Stephanie/Workbooks/DATA/gender.csv')

In [30]:
gender.head()

,Viewer gender,Views (%),Average view duration,Average percentage viewed (%),Watch time (hours) (%)
0,Female,42.59,0:02:57,30.76,45.34
1,Male,57.40,0:02:39,28.68,54.66
2,User-specified,0.00,0:00:25,2.57,0.00


In [31]:
content = pd.read_csv('/content/drive/MyDrive/BUS 4640 - Stephanie/Workbooks/DATA/content.csv')

In [32]:
content.head()

,Content,Video title,Video publish time,Views,Watch time (hours),Subscribers,Estimated revenue (USD),Average view duration,Impressions,Impressions click-through rate (%)
0,Total,NaN,NaN,1296398.0,61731.1045,4587.0,3899.310,0:02:51,8202554.0,6.46
1,KxBtMmzDRTk,Graphing Supply and Demand in Excel,"Jan 28, 2016",211830.0,8606.9686,282.0,1093.410,0:02:26,645165.0,8.37
2,R6H4Jy3fSDc,Calculating Marginal Revenue in Excel,"Nov 24, 2013",169731.0,5409.9523,117.0,233.009,0:01:54,249912.0,10.44
3,HZBfrgg1VXc,Capacity and Bottlenecks,"Jan 31, 2017",96891.0,3452.4370,573.0,433.318,0:02:08,795590.0,8.53
4,CBDfMETpaH4,Capacity Planning Process,"Jan 31, 2017",79221.0,3917.1554,388.0,453.487,0:02:58,705841.0,6.39


In [33]:
geog = pd.read_csv('/content/drive/MyDrive/BUS 4640 - Stephanie/Workbooks/DATA/geography.csv')

In [34]:
geog.head()

,Geography,Views,Watch time (hours),Average view duration
0,Total,1296398,61731.1045,0:02:51
1,US,327135,14272.6996,0:02:37
2,CA,159164,14084.2513,0:05:18
3,IN,151259,4751.7523,0:01:53
4,PH,86627,3936.1559,0:02:43


In [35]:
!pip install iso3166

In [36]:
from iso3166 import countries

In [37]:
geog_filtered = geog[geog['Geography'] != 'Total'].astype(str)
geog_filtered = geog_filtered[geog_filtered['Geography'] != 'nan']

country_names =[]

for country in geog_filtered['Geography']:
    c_name = countries.get(country)
    country_names.append(c_name[2])

geog_filtered['iso_alpha'] = country_names

In [38]:
geog_filtered.head()

,Geography,Views,Watch time (hours),Average view duration,iso_alpha
1,US,327135,14272.6996,0:02:37,USA
2,CA,159164,14084.2513,0:05:18,CAN
3,IN,151259,4751.7523,0:01:53,IND
4,PH,86627,3936.1559,0:02:43,PHL
5,GB,32053,1446.7714,0:02:42,GBR


In [39]:
geog_filtered["Views"] = geog_filtered["Views"].astype(int)

In [40]:
geog_filtered.to_csv('/content/drive/MyDrive/BUS 4640 - Stephanie/Workbooks/DATA/geogfiltered.csv', index=False)

## Full Dashboard from Youtube Analytics

In [41]:
!pip install -q gradio

In [45]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import gradio as gr

################################
# Load data

TotalViews = pd.read_csv('/content/drive/MyDrive/BUS 4640 - Stephanie/Workbooks/DATA/Totalviews.csv')
age = pd.read_csv('/content/drive/MyDrive/BUS 4640 - Stephanie/Workbooks/DATA/age.csv')
geogfiltered = pd.read_csv('/content/drive/MyDrive/BUS 4640 - Stephanie/Workbooks/DATA/geogfiltered.csv')
gender = pd.read_csv('/content/drive/MyDrive/BUS 4640 - Stephanie/Workbooks/DATA/gender.csv')
content = pd.read_csv('/content/drive/MyDrive/BUS 4640 - Stephanie/Workbooks/DATA/content.csv')
content = content.tail(-1)

################################
# Color choices for menu

color_theme_list = ['blues', 'cividis', 'greens', 'inferno', 'magma',
                    'plasma', 'reds', 'rainbow', 'turbo', 'viridis']
donut_colors = {'blue':   ['#29b5e8', '#155F7A'],
                'green':  ['#27AE60', '#12783D'],
                'orange': ['#F39C12', '#875A12'],
                'red':    ['#E74C3C', '#781F16']}

################################
# Plots

def make_choropleth(input_df, input_id, input_column, input_color_theme):
    fig = px.choropleth(input_df, locations=input_id, color=input_column,
                        color_continuous_scale=input_color_theme,
                        hover_name=input_id, locationmode='ISO-3')
    fig.update_layout(template='plotly_dark',
                      plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
                      margin=dict(l=0, r=0, t=0, b=0), height=350)
    return fig

def make_linechart(input_df, start_year, end_year):
    years = pd.to_datetime(input_df['Date']).dt.year
    df_range = input_df[years.between(start_year, end_year)]
    fig = px.line(df_range, x='Date', y='Views',
                  labels={'Date': 'Date', 'Views': 'Viewership'})
    fig.update_layout(margin=dict(l=0, r=0, t=10, b=0), height=300)
    return fig

def make_donut(value, color):
    colors = donut_colors[color]
    fig = go.Figure(go.Pie(values=[value, 100 - value], hole=0.65, sort=False,
                           marker_colors=colors, textinfo='none', hoverinfo='skip'))
    fig.update_layout(showlegend=False, height=160,
                      margin=dict(l=0, r=0, t=0, b=0),
                      paper_bgcolor='rgba(0,0,0,0)',
                      annotations=[dict(text=f'{value} %', showarrow=False,
                                        font=dict(size=20, color=colors[0]))])
    return fig

def make_metric(yearslide):
    years = pd.to_datetime(TotalViews['Date']).dt.year
    viewtotal = TotalViews.loc[years == yearslide, 'Views'].sum()
    if yearslide > 2013:
        viewtotal_before = TotalViews.loc[years == yearslide - 1, 'Views'].sum()
        delta = viewtotal - viewtotal_before
        delta_color = 'green' if delta >= 0 else 'red'
        delta_text = f"<span style='color:{delta_color}'>{'▲' if delta >= 0 else '▼'} {delta:,}</span>"
    else:
        delta_text = ''
    return f"**{yearslide}**\n\n<span style='font-size:2em'>{viewtotal:,}</span>\n\n{delta_text}"

################################
# Update visuals from sidebar choices

def update_dashboard(yearslide, start_year, end_year, agegroup, gendergroup,
                     selected_color_theme, selected_color):
    donut1 = gender.loc[gender['Viewer gender'] == gendergroup, 'Watch time (hours) (%)'].iloc[0]
    donut2 = age.loc[age['Viewer age'] == agegroup, 'Watch time (hours) (%)'].iloc[0]
    return (make_metric(yearslide),
            f'**{gendergroup}**', make_donut(donut1, selected_color),
            f'**{agegroup}**', make_donut(donut2, selected_color),
            make_choropleth(geogfiltered, 'iso_alpha', 'Views', selected_color_theme),
            make_linechart(TotalViews, start_year, end_year))

################################
# Layout

with gr.Blocks(title='Youtube Analytics') as demo:
    with gr.Row():
        # Sidebar
        with gr.Column(scale=2):
            gr.Markdown('## Analytics Dashboard')
            yearslide = gr.Slider(2013, 2024, value=2020, step=1, label='Yearly Data')
            start_year = gr.Slider(2013, 2024, value=2018, step=1, label='Year range: start')
            end_year = gr.Slider(2013, 2024, value=2022, step=1, label='Year range: end')
            agegroup = gr.Dropdown(list(age['Viewer age']), value=age['Viewer age'][0],
                                   label='Select an age group')
            gendergroup = gr.Dropdown(list(gender['Viewer gender']), value=gender['Viewer gender'][0],
                                      label='Select a gender')
            color_theme = gr.Dropdown(color_theme_list, value='blues', label='Select a color theme')
            color = gr.Dropdown(list(donut_colors), value='blue', label='Select a color')

        # Column 0: metric and donuts
        with gr.Column(scale=2):
            gr.Markdown('#### Number of Viewers')
            metric = gr.Markdown()
            gr.Markdown('#### Viewer Demographics')
            gender_label = gr.Markdown()
            donut_gender = gr.Plot(show_label=False)
            age_label = gr.Markdown()
            donut_age = gr.Plot(show_label=False)

        # Column 1: map and line chart
        with gr.Column(scale=5):
            gr.Markdown('#### Viewership')
            choropleth = gr.Plot(show_label=False)
            linechart = gr.Plot(show_label=False)

        # Column 2: top videos
        with gr.Column(scale=3):
            gr.Markdown('#### Top Videos')
            gr.Dataframe(content[['Video title', 'Views']], interactive=False)
            with gr.Accordion('About', open=True):
                gr.Markdown('- Data: Powers Youtube Channel')

    inputs = [yearslide, start_year, end_year, agegroup, gendergroup, color_theme, color]
    outputs = [metric, gender_label, donut_gender, age_label, donut_age, choropleth, linechart]

    # Run update_dashboard when the page loads and whenever any input changes
    gr.on(triggers=[demo.load] + [i.change for i in inputs],
          fn=update_dashboard, inputs=inputs, outputs=outputs)

demo.launch(share=True, height=1000)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e9112cd6a3d0f6dc20.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
